# H-005 · Size & Value Feature Suite

Factor test for **H-005** (equities): whether size / valuation levels, valuation rate-of-change, size momentum, and value–momentum interaction features carry cross-sectional predictive power for forward returns at Alphalens `periods=(1, 5, 21)` (primary narrative **5d**).

- **Idea** — Reconstruct daily `market_cap` / `pe` / `pb` from SEC Company Facts + closes (`fetch_size_value_daily`); screen a balanced window / momentum grid on research IS only.
- **Claim** — Value levels, valuation RoC, and value–momentum geometry improve next-week IC beyond size alone.
- **Why it might work** — Classic value (book / earnings yield); “getting cheaper/richer” (Δlog valuation) often predicts better at 5–21d than static levels; value and momentum are negatively correlated so interaction / distance / residual features can lift spreads.
- **Data** — Daily OHLCV long panel (`s1_factor_panel_train.parquet`) merged with `fetch_size_value_daily` on `["date", "ticker"]`.

## Features (8 store callers)

| Store caller | Output column(s) | `normalize` |
|---|---|---|
| `add_book_yield` | `book_yield` | True (CS pct-rank) |
| `add_earnings_yield` | `earnings_yield` | True |
| `add_log_mcap` | `log_mcap` | True |
| `add_valuation_roc` | `val_roc_{metric}` [`_{W}`] | **none** (never CS-ranked) |
| `add_size_momentum` | `size_mom` [`_{W}`] | **none** (never CS-ranked) |
| `add_value_momentum_interaction` | `val_mom_interact` | **none** (never CS-ranked) |
| `add_value_momentum_distance` | `val_mom_dist` | **none** (never CS-ranked) |
| `add_value_momentum_residual` | `val_mom_resid` | **none** (never CS-ranked) |

**Normalize policy:** CS pct-rank only for level features (yields, log mcap). RoC / size-mom / value–momentum features are already Δlog, rank-space, or z-scored — no second CS rank.

## Size/value feature parquet cache

| Path | Role |
|------|------|
| `01_data/data_files/s1_equities/s1_factor_panel_train.parquet` | Research IS OHLCV (cold path only) |
| `01_data/data_files/s1_equities/s1_h005_sv_panel.parquet` | Cached panel with H-005 columns |

**Save gate:** on a cold build, after features are built, write the SV parquet **before** any Alphalens screen / tear sheet.

**Invalidate** by deleting the SV parquet or setting `FORCE_REBUILD = True` when changing grid constants or store code.

## Sample discipline

- Do **not** use `s1_factor_panel_full.parquet` for window keep/kill.
- Overlapping 5d / 21d labels warrant purge/embargo in later walk-forward; this notebook screens IC on research IS only.
- Variant count = number of H-005 factor columns screened (see `EXPECTED_N_FACTORS`).


## 0. Imports & Config

Resolve the repo root; configure the parameter grid, SV-cache paths, and Alphalens periods. Set `FORCE_REBUILD = True` to ignore an existing SV parquet and rebuild from the train IS.


In [ ]:
from __future__ import annotations

import itertools
import os
import re
import sys
import time

# Jupyter cwd is often this notebook's folder, not the repo root; walk up until we find 01_data/ingestion.
ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import alphalens as al
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib.backends.backend_pdf import PdfPages

from data.ingestion.alternative_data.size_value_fetcher import fetch_size_value_daily
from data.processing.feature_implementation.size_and_valuation_features import (
    add_value_momentum_distance as add_val_mom_dist_raw,
    add_value_momentum_interaction as add_val_mom_interact_raw,
    add_value_momentum_residual as add_val_mom_resid_raw,
)
from data.processing.feature_store import (
    add_book_yield,
    add_earnings_yield,
    add_log_mcap,
    add_size_momentum,
    add_valuation_roc,
)

TRAIN_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_factor_panel_train.parquet"
)
SV_PANEL_PATH = os.path.join(
    ROOT, "01_data", "data_files", "s1_equities", "s1_h005_sv_panel.parquet"
)
TEARSHEET_DIR = os.path.join(
    ROOT, "02_research", "notebooks", "factor_tests", "tearsheets"
)

# --- Parameter grid (edit these lists) ---
WINDOWS = [21, 63, 126, 252]           # val_roc / size_mom windows
MOM_LOOKBACKS = [126, 252]             # value-momentum lookbacks
MOM_SKIPS = [10, 21, 42]               # value-momentum skips
REGRESSION_WINDOWS = [126, 252]        # val_mom_resid OLS windows
VAL_METRICS = ["pe", "pb"]             # valuation_roc metrics

EXPECTED_N_FACTORS = (
    3  # book_yield, earnings_yield, log_mcap
    + len(VAL_METRICS) * len(WINDOWS)  # val_roc_{metric}_{W}
    + len(WINDOWS)  # size_mom_{W}
    + 2 * len(MOM_LOOKBACKS) * len(MOM_SKIPS)  # interact + dist
    + len(REGRESSION_WINDOWS) * len(MOM_LOOKBACKS) * len(MOM_SKIPS)  # resid
)

# --- Fixed for this notebook ---
FORCE_REBUILD = False
FFILL_LIMIT = 5  # business days; avoid stale fundamentals
PERIODS = (1, 5, 21)  # primary narrative = 5d
QUANTILES = 5
MAX_LOSS = 0.35

H005_PREFIXES = (
    "book_yield",
    "earnings_yield",
    "log_mcap",
    "val_roc_",
    "size_mom",
    "val_mom_interact",
    "val_mom_dist",
    "val_mom_resid",
)

print(f"ROOT={ROOT}")
print(f"EXPECTED_N_FACTORS={EXPECTED_N_FACTORS}")
print(f"FORCE_REBUILD={FORCE_REBUILD}")


## 1. Data Loading

### Size/value parquet cache contract

1. If `FORCE_REBUILD` is False **and** `s1_h005_sv_panel.parquet` exists → **CACHE HIT**: load it into `panel` (features already present).
2. Otherwise → **CACHE MISS / cold build**: load `s1_factor_panel_train.parquet`, call `fetch_size_value_daily` (pass `price_panel` to avoid re-download), merge on `["date", "ticker"]`.
3. On a warm load, validate H-005 column count. If the cache looks stale (wrong count), fall through to a cold build.

Do **not** apply another 70/30 split here — the train parquet is already research IS.


In [ ]:
def _h005_factor_cols(frame: pd.DataFrame) -> list[str]:
    cols = []
    for c in frame.columns:
        if c in ("book_yield", "earnings_yield", "log_mcap"):
            cols.append(c)
        elif c == "size_mom" or c.startswith("size_mom_"):
            cols.append(c)
        elif c.startswith("val_roc_"):
            cols.append(c)
        elif c.startswith("val_mom_interact"):
            cols.append(c)
        elif c.startswith("val_mom_dist"):
            cols.append(c)
        elif c.startswith("val_mom_resid"):
            cols.append(c)
    return cols


use_sv_cache = (
    (not FORCE_REBUILD)
    and os.path.isfile(SV_PANEL_PATH)
)

if use_sv_cache:
    panel = pd.read_parquet(SV_PANEL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    FACTOR_COLS = _h005_factor_cols(panel)
    if len(FACTOR_COLS) != EXPECTED_N_FACTORS:
        print(
            f"CACHE STALE: found {len(FACTOR_COLS)} H-005 cols "
            f"(expected {EXPECTED_N_FACTORS}) - falling back to cold build"
        )
        use_sv_cache = False
    else:
        n_tickers = panel["ticker"].nunique()
        n_dates = panel["date"].nunique()
        print(
            f"CACHE HIT: {SV_PANEL_PATH}\n"
            f"rows={len(panel):,}  tickers={n_tickers}  dates={n_dates:,}  "
            f"factor cols={len(FACTOR_COLS)}  "
            f"[{panel['date'].min().date()} -> {panel['date'].max().date()}]"
        )

if not use_sv_cache:
    panel = pd.read_parquet(TRAIN_PANEL_PATH)
    panel["date"] = pd.to_datetime(panel["date"])
    panel["ticker"] = panel["ticker"].astype(str).str.strip().str.upper()
    tickers = sorted(panel["ticker"].unique().tolist())
    start = panel["date"].min().date()
    end = panel["date"].max().date()
    print(
        f"CACHE MISS: loaded {TRAIN_PANEL_PATH}\n"
        f"rows={len(panel):,}  tickers={len(tickers)}  "
        f"[{start} -> {end}]  (will fetch size/value + build features)"
    )
    sv = fetch_size_value_daily(
        tickers,
        start_date=start,
        end_date=end,
        price_panel=panel,
    )
    sv["date"] = pd.to_datetime(sv["date"])
    sv["ticker"] = sv["ticker"].astype(str).str.strip().str.upper()
    merge_cols = [
        "date", "ticker", "shares_outstanding", "book_equity", "eps_ttm",
        "market_cap", "pe", "pb",
    ]
    panel = panel.merge(sv[merge_cols], on=["date", "ticker"], how="left")
    print(
        f"SV merge: market_cap non-null={panel['market_cap'].notna().mean():.1%}  "
        f"pe={panel['pe'].notna().mean():.1%}  pb={panel['pb'].notna().mean():.1%}"
    )
    FACTOR_COLS = []

panel.head()


## 2. Data Cleaning & Engineering

Limited forward-fill of size/value columns within each ticker (`FFILL_LIMIT` business days) so short gaps between filings do not drop the whole cross-section. No floor / winsorize in library code — apply any extra cleaning here only.


In [ ]:
SV_COLS = ["market_cap", "pe", "pb", "shares_outstanding", "book_equity", "eps_ttm"]
present = [c for c in SV_COLS if c in panel.columns]

if not use_sv_cache:
    panel = panel.sort_values(["ticker", "date"]).copy()
    panel[present] = (
        panel.groupby("ticker", sort=False)[present]
        .ffill(limit=FFILL_LIMIT)
    )

print("NaN coverage after limited ffill (cold path only applies ffill):")
for c in present:
    print(f"  {c}: non-null={panel[c].notna().mean():.1%}")


## 3. Modeling / Signal Construction

Build all H-005 columns on the cold path. Level features use store defaults (`normalize=True`). Windowed / value–momentum features never CS-rank.

Value–momentum grids call raw panel helpers with explicit column names (store callers take single lookback/skip ints).

**Save gate (cold path):** after build, write `s1_h005_sv_panel.parquet` **before** Alphalens.


In [ ]:
t0 = time.perf_counter()

if use_sv_cache:
    print("Skipping §3 rebuild - using cached H-005 columns")
    FACTOR_COLS = _h005_factor_cols(panel)
else:
    # Level features (CS-ranked by default)
    panel = add_book_yield(panel)
    panel = add_earnings_yield(panel)
    panel = add_log_mcap(panel)

    # Windowed (never CS-ranked)
    for metric in VAL_METRICS:
        panel = add_valuation_roc(panel, metric=metric, window=WINDOWS)
    panel = add_size_momentum(panel, window=WINDOWS)

    # Value-momentum grids (never second-ranked)
    for L, S in itertools.product(MOM_LOOKBACKS, MOM_SKIPS):
        if L <= S:
            raise ValueError(f"mom_lookback must be > mom_skip, got L={L}, S={S}")
        panel = add_val_mom_interact_raw(
            panel, mom_lookback=L, mom_skip=S, col=f"val_mom_interact_{L}_{S}"
        )
        panel = add_val_mom_dist_raw(
            panel, mom_lookback=L, mom_skip=S, col=f"val_mom_dist_{L}_{S}"
        )

    for RW, L, S in itertools.product(REGRESSION_WINDOWS, MOM_LOOKBACKS, MOM_SKIPS):
        if L <= S:
            raise ValueError(f"mom_lookback must be > mom_skip, got L={L}, S={S}")
        panel = add_val_mom_resid_raw(
            panel,
            regression_window=RW,
            mom_lookback=L,
            mom_skip=S,
            col=f"val_mom_resid_{RW}_{L}_{S}",
        )

    FACTOR_COLS = _h005_factor_cols(panel)
    assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
        f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
    )

    os.makedirs(os.path.dirname(SV_PANEL_PATH), exist_ok=True)
    panel.to_parquet(SV_PANEL_PATH, index=False)
    print(f"Wrote {SV_PANEL_PATH}")

elapsed = time.perf_counter() - t0
print(f"H-005 factor columns: {len(FACTOR_COLS)}  (build/load wall={elapsed:.1f}s)")
assert len(FACTOR_COLS) == EXPECTED_N_FACTORS, (
    f"expected {EXPECTED_N_FACTORS} factor cols, got {len(FACTOR_COLS)}"
)
print("Factor columns:")
for c in sorted(FACTOR_COLS):
    print(f"  {c}")


## 4. Evaluation

Screen every H-005 column at `periods=(1, 5, 21)` with `quantiles=5`. Primary sort key: **`ic_5d`**. Decode column parameters with `parse_sv_factor_name`.

Full tear sheets are **PDF-only** (no inline display) — edit `TEAR_FACTORS` after reviewing §4.1.


In [ ]:
def to_alphalens_prices(panel: pd.DataFrame) -> pd.DataFrame:
    """Wide close matrix for Alphalens only (dates x tickers)."""
    prices = panel.pivot(index="date", columns="ticker", values="close")
    prices.index = pd.to_datetime(prices.index)
    return prices.sort_index()


def to_alphalens_factor(panel: pd.DataFrame, col: str) -> pd.Series:
    """MultiIndex (date, ticker) factor series for Alphalens."""
    factor = panel.set_index(["date", "ticker"])[col].dropna()
    factor.index = factor.index.set_levels(
        pd.to_datetime(factor.index.levels[0]), level=0
    )
    return factor.sort_index()


def _period_label(period_index: pd.Index, period: int, position: int):
    """Match Alphalens period label ('1D', '5D', ...) or fall back by position."""
    for c in (f"{period}D", f"{period}d", period, str(period)):
        if c in period_index:
            return c
    return period_index[position]


def parse_sv_factor_name(col: str) -> dict | None:
    """Decode an H-005 factor column into family / window metadata."""
    if col in ("book_yield", "earnings_yield", "log_mcap"):
        return {"feature": col, "family": col}

    m = re.fullmatch(r"val_roc_(pe|pb)_(\d+)", col)
    if m:
        return {
            "feature": f"val_roc_{m.group(1)}",
            "family": f"val_roc_{m.group(1)}",
            "metric": m.group(1),
            "window": int(m.group(2)),
        }
    m = re.fullmatch(r"val_roc_(pe|pb)", col)
    if m:
        return {
            "feature": f"val_roc_{m.group(1)}",
            "family": f"val_roc_{m.group(1)}",
            "metric": m.group(1),
        }

    m = re.fullmatch(r"size_mom_(\d+)", col)
    if m:
        return {"feature": "size_mom", "family": "size_mom", "window": int(m.group(1))}
    if col == "size_mom":
        return {"feature": "size_mom", "family": "size_mom"}

    m = re.fullmatch(r"val_mom_interact_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_interact",
            "family": "val_mom_interact",
            "lookback": int(m.group(1)),
            "skip": int(m.group(2)),
        }
    if col == "val_mom_interact":
        return {"feature": "val_mom_interact", "family": "val_mom_interact"}

    m = re.fullmatch(r"val_mom_dist_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_dist",
            "family": "val_mom_dist",
            "lookback": int(m.group(1)),
            "skip": int(m.group(2)),
        }
    if col == "val_mom_dist":
        return {"feature": "val_mom_dist", "family": "val_mom_dist"}

    m = re.fullmatch(r"val_mom_resid_(\d+)_(\d+)_(\d+)", col)
    if m:
        return {
            "feature": "val_mom_resid",
            "family": "val_mom_resid",
            "reg_window": int(m.group(1)),
            "lookback": int(m.group(2)),
            "skip": int(m.group(3)),
        }
    if col == "val_mom_resid":
        return {"feature": "val_mom_resid", "family": "val_mom_resid"}

    return None


def factor_screen_metrics(
    factor: pd.Series,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
) -> dict:
    """Mean IC and Q5-Q1 mean return spread for each forward period."""
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=factor,
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )
    mean_ic = al.performance.mean_information_coefficient(factor_data)
    mean_ret, _ = al.performance.mean_return_by_quantile(factor_data, demeaned=True)

    row = {}
    for i, p in enumerate(periods):
        ic_key = _period_label(mean_ic.index, p, i)
        ret_key = _period_label(mean_ret.columns, p, i)
        row[f"ic_{p}d"] = float(mean_ic.loc[ic_key])
        q_hi, q_lo = mean_ret.index.max(), mean_ret.index.min()
        row[f"spread_{p}d"] = float(
            mean_ret.loc[q_hi, ret_key] - mean_ret.loc[q_lo, ret_key]
        )
    return row


def run_full_tear(
    panel: pd.DataFrame,
    factor_col: str,
    prices: pd.DataFrame,
    *,
    periods: tuple[int, ...] = PERIODS,
    quantiles: int = QUANTILES,
    max_loss: float = MAX_LOSS,
    tearsheet_dir: str = TEARSHEET_DIR,
):
    """Build factor_data, run Alphalens full tear, save figs to multi-page PDF.

    PDF-only: Agg backend + patched ``plt.show`` so figures are not displayed
    inline (many H-005 variants).
    """
    if factor_col not in panel.columns:
        raise ValueError(
            f"{factor_col!r} not in panel - pick a screened column "
            f"(available: {FACTOR_COLS})"
        )
    plt.close("all")
    plt.ioff()
    factor_data = al.utils.get_clean_factor_and_forward_returns(
        factor=to_alphalens_factor(panel, factor_col),
        prices=prices,
        quantiles=quantiles,
        periods=periods,
        max_loss=max_loss,
    )

    os.makedirs(tearsheet_dir, exist_ok=True)
    out_path = os.path.join(tearsheet_dir, f"H-005_{factor_col}.pdf")
    pdf = PdfPages(out_path)
    n_pages = 0
    _original_show = plt.show

    def _show_and_savefig(*args, **kwargs):
        nonlocal n_pages
        for num in list(plt.get_fignums()):
            fig = plt.figure(num)
            if fig.axes:
                pdf.savefig(fig, bbox_inches="tight")
                n_pages += 1
        plt.close("all")

    plt.show = _show_and_savefig
    try:
        al.tears.create_full_tear_sheet(factor_data, long_short=True)
        _show_and_savefig()
    finally:
        plt.show = _original_show
        pdf.close()
        plt.close("all")

    print(f"Wrote {out_path} ({n_pages} pages)")
    return factor_data


### 4.1 Window screen summary

Full IC / spread table sorted by `ic_5d`, plus a **best-by-family** table (one row per feature stem).

**Expected signs (literature / intuition):**
- `book_yield` / `earnings_yield` — positive IC (cheap → long)
- `log_mcap` — often negative IC (size); weak in large-cap sleeves
- `val_roc_*` — context-dependent (“getting cheaper” vs richer)
- `size_mom` — overlaps price momentum; test incremental IC
- `val_mom_interact` — high value × high mom; often positive
- `val_mom_dist` — distance from ideal (1,1); often **negative** IC (closer = better)
- `val_mom_resid` — orthogonal value after mom; sign empirical


In [ ]:
t0 = time.perf_counter()
prices = to_alphalens_prices(panel)
rows = []
for col in FACTOR_COLS:
    meta = parse_sv_factor_name(col) or {}
    metrics = factor_screen_metrics(to_alphalens_factor(panel, col), prices)
    rows.append({"factor": col, **meta, **metrics})

summary = (
    pd.DataFrame(rows)
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)
print(f"Alphalens screen wall={time.perf_counter() - t0:.1f}s  n={len(summary)}")

summary["feature_family"] = summary["factor"].map(
    lambda c: (parse_sv_factor_name(c) or {}).get("family")
)
best_by_family = (
    summary.sort_values("ic_5d", ascending=False)
    .groupby("feature_family", as_index=False)
    .first()
    .sort_values("ic_5d", ascending=False)
    .reset_index(drop=True)
)

print("\n=== Best by feature family (ic_5d) ===")
print(best_by_family.to_string())

print("\n=== Full screen (sorted by ic_5d) ===")
with pd.option_context("display.max_columns", None, "display.max_rows", None):
    display(summary)


### 4.2 Full tear sheets (manual selection)

Edit `TEAR_FACTORS` after reviewing §4.1. Each tear is saved as a multi-page PDF under:

`02_research/notebooks/factor_tests/tearsheets/H-005_{factor_col}.pdf`

**No inline display** (runtime / notebook size). Re-running overwrites the same paths. **Do not** tear all screened columns.


In [ ]:
# Edit after reviewing §4.1 (defaults are placeholders)
TEAR_FACTORS = [
    "book_yield",
    "val_roc_pb_63",
    "val_mom_dist_252_21",
]

for tear_col in TEAR_FACTORS:
    print(f"\n===== Tear sheet: {tear_col} =====")
    run_full_tear(panel, tear_col, prices)


## 5. Wrap-up / next steps

- Record keep/kill decisions in `02_research/hypothesis_log.md` (H-005 Alphalens summary).
- Holdout: reserved; do not peek at `s1_factor_panel_full.parquet` for keep/kill.
- Tear PDFs under `02_research/notebooks/factor_tests/tearsheets/` named `H-005_{factor_col}.pdf`.
- If cold-run SEC fetch dominates wall time, keep `FORCE_REBUILD = False` after the first successful build.
- If screening wall time is high, trim `WINDOWS` / `MOM_SKIPS` / `REGRESSION_WINDOWS` in §0 and rebuild.
- Next: walk-forward / purged CV before any `feature_spec` freeze.
